# CycleGAN Horse ↔ Zebra: App
Run all cells in Colab (a GPU is optional).

1. Upload your final `cyclegan_output` folder to Drive as `cyclegan_output_final`.
2. Run all cells. The last one prints a public link.

New: the app now accepts any image size and tells you the exact size of the result.

In [ ]:
!pip install -q gradio

## 1. Setup and weights

In [ ]:
import os, glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from google.colab import drive
drive.mount("/content/drive")

# Use the first folder that exists. Put your FINAL 100-epoch results in cyclegan_output_final.
CANDIDATES = ["/content/drive/MyDrive/cyclegan_output_final",
              "/content/drive/MyDrive/cyclegan_output"]
OUTPUT_DIR = next((d for d in CANDIDATES if os.path.exists(f"{d}/checkpoints/G.weights.h5")), None)
assert OUTPUT_DIR, "No weights found. Upload the cyclegan_output folder to My Drive."
print("Using:", OUTPUT_DIR)
print(os.listdir(f"{OUTPUT_DIR}/checkpoints"))

## 2. Layers and generator (accepts any size)

In [ ]:
class InstanceNorm(layers.Layer):
    """Instance normalization (normalizes each image/channel independently).
    Written by hand because tensorflow_addons is deprecated."""

    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.scale = self.add_weight(
            name="scale", shape=input_shape[-1:],
            initializer=keras.initializers.RandomNormal(1.0, 0.02), trainable=True)
        self.offset = self.add_weight(
            name="offset", shape=input_shape[-1:],
            initializer="zeros", trainable=True)

    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        x = (x - mean) * tf.math.rsqrt(var + self.epsilon)
        return self.scale * x + self.offset


class ReflectionPad2D(layers.Layer):
    """Pads borders by mirroring pixels, which avoids edge artifacts."""

    def __init__(self, pad=1, **kwargs):
        super().__init__(**kwargs)
        self.pad = pad

    def call(self, x):
        p = self.pad
        return tf.pad(x, [[0, 0], [p, p], [p, p], [0, 0]], mode="REFLECT")

In [ ]:
INIT = keras.initializers.RandomNormal(mean=0.0, stddev=0.02)


def residual_block(x, filters):
    skip = x
    x = ReflectionPad2D(1)(x)
    x = layers.Conv2D(filters, 3, kernel_initializer=INIT, use_bias=False)(x)
    x = InstanceNorm()(x)
    x = layers.ReLU()(x)
    x = ReflectionPad2D(1)(x)
    x = layers.Conv2D(filters, 3, kernel_initializer=INIT, use_bias=False)(x)
    x = InstanceNorm()(x)
    return layers.Add()([skip, x])


def build_generator(name, n_res_blocks=9):
    inp = layers.Input(shape=(None, None, 3))  # any size (multiple of 4)

    # c7s1-64
    x = ReflectionPad2D(3)(inp)
    x = layers.Conv2D(64, 7, kernel_initializer=INIT, use_bias=False)(x)
    x = InstanceNorm()(x)
    x = layers.ReLU()(x)

    # Downsample: 128, 256
    for f in (128, 256):
        x = layers.Conv2D(f, 3, strides=2, padding="same",
                          kernel_initializer=INIT, use_bias=False)(x)
        x = InstanceNorm()(x)
        x = layers.ReLU()(x)

    # Residual blocks
    for _ in range(n_res_blocks):
        x = residual_block(x, 256)

    # Upsample: 128, 64
    for f in (128, 64):
        x = layers.Conv2DTranspose(f, 3, strides=2, padding="same",
                                   kernel_initializer=INIT, use_bias=False)(x)
        x = InstanceNorm()(x)
        x = layers.ReLU()(x)

    # Output layer
    x = ReflectionPad2D(3)(x)
    x = layers.Conv2D(3, 7, activation="tanh", kernel_initializer=INIT)(x)
    return keras.Model(inp, x, name=name)

## 3. Load trained generators

In [ ]:
gen_G = build_generator("G_horse_to_zebra")   # horse -> zebra
gen_F = build_generator("F_zebra_to_horse")   # zebra -> horse
gen_G.load_weights(f"{OUTPUT_DIR}/checkpoints/G.weights.h5")
gen_F.load_weights(f"{OUTPUT_DIR}/checkpoints/F.weights.h5")
try:
    EPOCHS_DONE = int(open(f"{OUTPUT_DIR}/checkpoints/last_epoch.txt").read())
except Exception:
    EPOCHS_DONE = None
print("Generators loaded, epochs trained:", EPOCHS_DONE)

## 4. Launch the app

In [ ]:
import gradio as gr

DIRECTIONS = ["Horse → Zebra", "Zebra → Horse"]
SIZE_MODES = ["Keep my image size", "256 x 256 (trained size)"]


def pick_size(w, h, mode, max_side):
    """Size the model will work at. Always a multiple of 4 (the network halves twice)."""
    if mode == SIZE_MODES[1]:
        return 256, 256
    scale = min(1.0, max_side / max(w, h))          # shrink big images, never enlarge
    snap = lambda v: max(64, int(round(v * scale / 4)) * 4)
    return snap(w), snap(h)


def translate(img, direction, mode, max_side, back_to_original):
    if img is None:
        return None, "Upload an image to start."
    gen = gen_G if direction == DIRECTIONS[0] else gen_F
    h, w = img.shape[:2]
    nw, nh = pick_size(w, h, mode, max_side)
    x = tf.cast(img[..., :3], tf.float32)
    x = tf.image.resize(x, [nh, nw], antialias=True) / 127.5 - 1.0
    y = gen(x[None, ...], training=False)[0].numpy()
    out = ((y + 1.0) * 127.5).clip(0, 255).astype("uint8")
    note = f"Input {w} x {h} px. The model worked at {nw} x {nh} px."
    if back_to_original and (nw, nh) != (w, h):
        out = np.asarray(Image.fromarray(out).resize((w, h), Image.LANCZOS))
        note += f" Result stretched back to {w} x {h} px (may look softer)."
    else:
        note += f" Result is {out.shape[1]} x {out.shape[0]} px."
    if mode == SIZE_MODES[1] and (w, h) != (256, 256):
        note += " Your image was squashed to a square, so proportions may look different."
    return out, note


sample_files = sorted(glob.glob(f"{OUTPUT_DIR}/samples/epoch_*.png"))
gallery_files = []
if sample_files:
    idx = np.linspace(0, len(sample_files) - 1, min(6, len(sample_files))).astype(int)
    gallery_files = [sample_files[i] for i in idx]
loss_img = f"{OUTPUT_DIR}/loss_curves.png"
epochs_txt = f"{EPOCHS_DONE} epochs" if EPOCHS_DONE else "many epochs"

with gr.Blocks(title="CycleGAN Horse ↔ Zebra") as demo:
    gr.Markdown("# 🐎 ↔ 🦓 CycleGAN: Horse ↔ Zebra\nUpload a photo and watch it change species.")

    with gr.Tab("Try it"):
        direction = gr.Radio(DIRECTIONS, value=DIRECTIONS[0], label="Direction")
        with gr.Row():
            mode = gr.Radio(SIZE_MODES, value=SIZE_MODES[0], label="Output size")
            max_side = gr.Slider(256, 1024, value=512, step=64, label="Largest side (px) when keeping my size")
        back = gr.Checkbox(value=False, label="Stretch the result back to my original size")
        with gr.Row():
            inp = gr.Image(label="Your image", type="numpy")
            out = gr.Image(label="Translated image")
        info = gr.Textbox(label="Sizes", interactive=False)
        gr.Button("Translate", variant="primary").click(
            translate, [inp, direction, mode, max_side, back], [out, info])
        with gr.Accordion("Which image sizes work?", open=False):
            gr.Markdown("""
- The model was **trained on 256 x 256** photos, so it is most reliable near that size.
- **Keep my image size** (default): your image is used as it is, up to the limit you choose. Larger photos are shrunk to that limit to keep things fast. The result has the same shape as the image the model worked on, with each side a multiple of 4.
- **256 x 256**: the safest quality, but the image is squashed to a square.
- Photos much larger than 512 px can get finer, patchier stripes, because the model learned stripe size at 256 px.
- JPG, PNG and WEBP all work. Clear photos of one horse or zebra give the best results.
""")

    with gr.Tab("Training progress"):
        gr.Markdown("Samples saved during training (real → fake → cycled), from early to late epochs.")
        if gallery_files:
            gr.Gallery(value=gallery_files, columns=2, height="auto", label="Epoch samples")
        if os.path.exists(loss_img):
            gr.Image(value=loss_img, label="Loss curves", interactive=False)

    with gr.Tab("About"):
        gr.Markdown(f"""
**Model:** CycleGAN with two ResNet generators (9 residual blocks) and two 70x70 PatchGAN discriminators, trained for {epochs_txt}.

**Losses:** least-squares adversarial + cycle-consistency (lambda = 10) + identity (lambda = 5).

**Data:** Horse2Zebra, about 1,067 horse and 1,334 zebra training images at 256x256.

**Framework:** TensorFlow / Keras.
""")

demo.launch(share=True, debug=False)